# IA — Aula 5 — Aprendizado de Máquina Clássico
## Trabalho da semana: Filtro de Spam — Spambase (UCI)

**T5-A1 — Ambiente, carga e inspeção da base**

Objetivos desta etapa:

1. Preparar o ambiente de execução.
2. Carregar a base **Spambase (UCI)**.
3. Confirmar as dimensões esperadas: **4.601 registros e 57 atributos preditores**.
4. Inspecionar a variável-alvo.
5. Verificar distribuição das classes, valores ausentes e tipos.
6. Confirmar a existência dos atributos `george`, `650`, `hp` e `hpl`, que serão usados em uma etapa posterior do trabalho.

> Nesta etapa **não treinaremos modelos ainda**. O objetivo é garantir que a base e o pipeline experimental estejam corretos antes dos experimentos.


## 1. Configuração do ambiente

A célula abaixo instala as dependências necessárias no ambiente atual.

> Em Google Colab ou Jupyter, execute esta célula uma vez. Se os pacotes já estiverem instalados, a operação será rápida.


In [ ]:
%pip install -q ucimlrepo pandas numpy scikit-learn matplotlib


## 2. Imports e configurações de reprodutibilidade

Usaremos `random_state = 42` nas próximas etapas para tornar os experimentos reproduzíveis.


In [ ]:
from __future__ import annotations

import sys
import platform
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo

RANDOM_STATE = 42

print("Python:", sys.version.split()[0])
print("Plataforma:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("random_state:", RANDOM_STATE)


## 3. Carregamento da Spambase

A base será obtida pelo pacote `ucimlrepo`, usando o identificador **94** da UCI Machine Learning Repository.


In [ ]:
spambase = fetch_ucirepo(id=94)

X = spambase.data.features.copy()
y = spambase.data.targets.squeeze().copy()

# Garantir rótulo inteiro (0 = legítimo, 1 = spam)
y = pd.to_numeric(y, errors="raise").astype(int)
y.name = "spam"

print("Dataset carregado com sucesso.")
print("X:", X.shape)
print("y:", y.shape)


## 4. Validação estrutural da base

Segundo o enunciado do trabalho, esperamos:

- **4.601 e-mails**
- **57 atributos numéricos**
- alvo binário: **0 = legítimo**, **1 = spam**

As asserções abaixo fazem o notebook interromper a execução caso a estrutura recebida seja diferente da esperada.


In [ ]:
EXPECTED_ROWS = 4601
EXPECTED_FEATURES = 57

assert X.shape[0] == EXPECTED_ROWS, (
    f"Número inesperado de registros: {X.shape[0]} (esperado: {EXPECTED_ROWS})"
)
assert X.shape[1] == EXPECTED_FEATURES, (
    f"Número inesperado de atributos: {X.shape[1]} (esperado: {EXPECTED_FEATURES})"
)
assert len(y) == EXPECTED_ROWS, (
    f"Tamanho inesperado do alvo: {len(y)} (esperado: {EXPECTED_ROWS})"
)
assert set(y.unique()).issubset({0, 1}), (
    f"Alvo não binário como esperado. Valores encontrados: {sorted(y.unique())}"
)

print("✓ 4.601 registros confirmados")
print("✓ 57 atributos preditores confirmados")
print("✓ alvo binário confirmado")


## 5. Primeira inspeção dos dados


In [ ]:
display(X.head())

print("\nTipos dos atributos:")
display(X.dtypes.value_counts().rename("quantidade").to_frame())

print("\nPrimeiros nomes de atributos:")
print(X.columns[:15].tolist())


## 6. Distribuição da variável-alvo

Uma das primeiras verificações em classificação é entender o balanço das classes.

Nesta base, esperamos aproximadamente **39% spam** e **61% e-mails legítimos**.

Essa distribuição será importante posteriormente para:

- construir o baseline;
- usar divisão estratificada;
- interpretar a acurácia corretamente;
- avaliar o custo dos falsos positivos.


In [ ]:
class_counts = y.value_counts().sort_index()
class_pct = y.value_counts(normalize=True).sort_index().mul(100)

class_summary = pd.DataFrame({
    "classe": ["legítimo", "spam"],
    "quantidade": [class_counts.get(0, 0), class_counts.get(1, 0)],
    "percentual": [class_pct.get(0, 0), class_pct.get(1, 0)],
}, index=[0, 1])

class_summary.index.name = "rotulo"
display(class_summary)

spam_rate = class_pct.get(1, np.nan)
legit_rate = class_pct.get(0, np.nan)

print(f"Legítimos: {legit_rate:.2f}%")
print(f"Spam:      {spam_rate:.2f}%")


## 7. Qualidade básica dos dados

Antes de qualquer modelagem, verificamos:

- valores ausentes;
- infinitos;
- duplicatas;
- atributos constantes.

Essas verificações são simples, mas ajudam a evitar conclusões erradas nas etapas seguintes.


In [ ]:
missing_total = int(X.isna().sum().sum())
inf_total = int(np.isinf(X.to_numpy(dtype=float)).sum())
duplicate_rows = int(pd.concat([X, y], axis=1).duplicated().sum())
constant_features = X.columns[X.nunique(dropna=False) <= 1].tolist()

quality_summary = pd.DataFrame({
    "checagem": [
        "valores ausentes",
        "valores infinitos",
        "linhas duplicadas (X + y)",
        "atributos constantes",
    ],
    "quantidade": [
        missing_total,
        inf_total,
        duplicate_rows,
        len(constant_features),
    ],
})

display(quality_summary)

if constant_features:
    print("Atributos constantes:", constant_features)


## 8. Estatísticas descritivas

As 57 variáveis são numéricas, mas não necessariamente estão na mesma escala.

Isso será especialmente importante no experimento com **k-NN**, porque esse algoritmo depende de distância entre observações.


In [ ]:
summary = X.describe().T
summary["range"] = summary["max"] - summary["min"]

display(summary.head(15))

print("\nAtributos com maior amplitude:")
display(
    summary[["min", "max", "mean", "std", "range"]]
    .sort_values("range", ascending=False)
    .head(10)
)


## 9. Verificação dos quatro atributos do experimento de generalização

O item 5 do trabalho exige retirar os atributos associados a:

- `george`
- `650`
- `hp`
- `hpl`

Na UCI, os nomes completos esperados são:

- `word_freq_george`
- `word_freq_650`
- `word_freq_hp`
- `word_freq_hpl`


In [ ]:
SPECIAL_FEATURES = [
    "word_freq_george",
    "word_freq_650",
    "word_freq_hp",
    "word_freq_hpl",
]

special_feature_check = pd.DataFrame({
    "atributo": SPECIAL_FEATURES,
    "presente": [feature in X.columns for feature in SPECIAL_FEATURES],
})

display(special_feature_check)

missing_special = [
    feature for feature in SPECIAL_FEATURES
    if feature not in X.columns
]

assert not missing_special, (
    "Atributos esperados não encontrados: "
    + ", ".join(missing_special)
)

print("✓ Os quatro atributos necessários para o experimento 5 estão presentes.")


## 10. Metadados do dataset

Esta célula registra informações fornecidas pelo repositório UCI. É útil para documentar a proveniência da base no notebook.


In [ ]:
print("METADADOS")
print("-" * 80)
print(spambase.metadata)

print("\nVARIÁVEIS")
print("-" * 80)
display(spambase.variables)


## 11. Checklist da T5-A1

Ao finalizar esta etapa, devemos ter confirmado:

- [x] ambiente configurado;
- [x] Spambase carregada;
- [x] 4.601 registros;
- [x] 57 atributos preditores;
- [x] alvo binário;
- [x] distribuição das classes conhecida;
- [x] qualidade básica dos dados inspecionada;
- [x] escalas dos atributos inspecionadas;
- [x] `george`, `650`, `hp` e `hpl` localizados.

### Próxima etapa — T5-A2

Construir o **baseline que classifica todo e-mail como legítimo** e calcular sua acurácia e demais métricas relevantes.


---
## T5-A2 — Baseline: sempre prever e-mail legítimo

O primeiro item do trabalho pede um **baseline que sempre prevê `"legítimo"`**.

O baseline estabelece uma referência mínima para os modelos que serão avaliados posteriormente. Como a classe majoritária da Spambase é `0 = legítimo`, um classificador que ignore completamente os atributos e sempre produza `0` terá uma acurácia igual à proporção de e-mails legítimos da base.

Entretanto, **acurácia isolada pode ser enganosa**: esse baseline não identifica nenhum spam. Por isso, além da acurácia, calcularemos a matriz de confusão, precision, recall e F1 para a classe positiva (`spam = 1`).


### 12. Imports para avaliação

Nesta etapa não há treinamento. Apenas construiremos uma predição constante e mediremos seu comportamento.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)


### 13. Construção do baseline

O vetor `y_baseline` terá o mesmo tamanho de `y`, mas todas as previsões serão `0`, isto é, **legítimo**.


In [ ]:
# Baseline: classificar todos os e-mails como legítimos (classe 0)
y_baseline = np.zeros(len(y), dtype=int)

assert len(y_baseline) == len(y)
assert set(np.unique(y_baseline)) == {0}

print(f"Total de previsões: {len(y_baseline)}")
print("Classe prevista pelo baseline: 0 = legítimo")


### 14. Métricas do baseline

A classe positiva do problema é `1 = spam`.

Para evitar avisos ao calcular precision quando o modelo não prevê nenhum spam, utilizamos `zero_division=0`.


In [ ]:
baseline_accuracy = accuracy_score(y, y_baseline)
baseline_precision = precision_score(y, y_baseline, pos_label=1, zero_division=0)
baseline_recall = recall_score(y, y_baseline, pos_label=1, zero_division=0)
baseline_f1 = f1_score(y, y_baseline, pos_label=1, zero_division=0)

baseline_results = pd.DataFrame({
    "modelo": ["Baseline — sempre legítimo"],
    "accuracy": [baseline_accuracy],
    "precision_spam": [baseline_precision],
    "recall_spam": [baseline_recall],
    "f1_spam": [baseline_f1],
})

display(
    baseline_results.style.format({
        "accuracy": "{:.4f}",
        "precision_spam": "{:.4f}",
        "recall_spam": "{:.4f}",
        "f1_spam": "{:.4f}",
    })
)

print(f"Acurácia do baseline: {baseline_accuracy:.2%}")
print(f"Recall de spam:       {baseline_recall:.2%}")


### 15. Matriz de confusão

Usaremos a convenção do `scikit-learn`:

\[
\begin{bmatrix}
TN & FP \\
FN & TP
\end{bmatrix}
\]

onde:

- **TN**: legítimo corretamente classificado como legítimo;
- **FP**: legítimo incorretamente classificado como spam;
- **FN**: spam incorretamente classificado como legítimo;
- **TP**: spam corretamente classificado como spam.


In [ ]:
cm = confusion_matrix(y, y_baseline, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

confusion_df = pd.DataFrame(
    cm,
    index=["Real: legítimo (0)", "Real: spam (1)"],
    columns=["Previsto: legítimo (0)", "Previsto: spam (1)"],
)

display(confusion_df)

print(f"TN (legítimos corretamente aceitos): {tn}")
print(f"FP (legítimos marcados como spam):   {fp}")
print(f"FN (spams que passam como legítimos): {fn}")
print(f"TP (spams detectados):                {tp}")


### 16. Relatório de classificação

Este relatório explicita por que a acurácia, sozinha, não é suficiente para avaliar o filtro.


In [ ]:
print(
    classification_report(
        y,
        y_baseline,
        labels=[0, 1],
        target_names=["legítimo", "spam"],
        zero_division=0,
    )
)


### 17. Interpretação do resultado

O baseline obtém uma acurácia equivalente à proporção da classe majoritária (`legítimo`). Apesar disso, seu desempenho como **filtro de spam é nulo**, pois:

- não prevê nenhum e-mail como spam;
- `recall` da classe spam = 0;
- todos os spams são falsos negativos;
- nenhum spam é efetivamente bloqueado.

Essa etapa mostra por que **acurácia não deve ser analisada isoladamente em problemas de classificação**, especialmente quando as classes são desbalanceadas ou quando os diferentes tipos de erro possuem custos distintos.

O baseline será usado como referência mínima para os modelos avaliados nas próximas etapas.


### Checkpoint T5-A2

Ao executar esta etapa no **Google Colab**, registre os valores obtidos. Eles serão reutilizados no relatório final de uma página.

**Próxima etapa — T5-A3:** criar uma divisão **train/test estratificada**, preservar um conjunto de teste final e configurar a **validação cruzada estratificada de 5 dobras** para os modelos do trabalho.


---
## T5-A3 — Split estratificado + StratifiedKFold de 5 dobras

Nesta etapa será preparada a infraestrutura experimental usada na comparação dos modelos.

A estratégia adotada será:

1. separar a base original em **treino (80%)** e **teste final (20%)**, preservando a proporção entre spam e e-mails legítimos por meio de `stratify=y`;
2. manter o conjunto de teste **isolado**, sem utilizá-lo para seleção de modelos ou ajuste de hiperparâmetros;
3. executar a comparação dos modelos somente no conjunto de treino por meio de **validação cruzada estratificada de 5 dobras** (`StratifiedKFold`).

Essa organização evita que o conjunto de teste final influencie as decisões tomadas durante o desenvolvimento do modelo.


### 18. Imports para divisão e validação cruzada

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

### 19. Divisão treino/teste estratificada

Usaremos:

- `test_size=0.20`
- `random_state=42`
- `stratify=y`

A estratificação garante que as proporções das classes permaneçam aproximadamente iguais nos dois subconjuntos.


In [ ]:
TEST_SIZE = 0.20

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Dimensões:")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")


### 20. Verificação da estratificação

Vamos comparar a proporção de spam na base completa, no conjunto de treino e no conjunto de teste.


In [ ]:
def class_distribution(target: pd.Series) -> pd.Series:
    return target.value_counts(normalize=True).sort_index().mul(100)

distribution_summary = pd.DataFrame({
    "base_completa_%": class_distribution(y),
    "treino_%": class_distribution(y_train),
    "teste_%": class_distribution(y_test),
}).rename(index={0: "legítimo (0)", 1: "spam (1)"})

display(distribution_summary.style.format("{:.2f}%"))


### 21. Verificações de integridade do split

As asserções abaixo garantem que:

- treino e teste não compartilham índices;
- a soma das amostras de treino e teste corresponde à base original;
- ambos os subconjuntos contêm as duas classes.


In [ ]:
assert set(X_train.index).isdisjoint(set(X_test.index)), (
    "Há sobreposição de índices entre treino e teste."
)

assert len(X_train) + len(X_test) == len(X), (
    "A soma de treino e teste não corresponde ao tamanho da base original."
)

assert set(y_train.unique()) == {0, 1}, "Treino não contém as duas classes."
assert set(y_test.unique()) == {0, 1}, "Teste não contém as duas classes."

print("✓ treino e teste sem sobreposição de índices")
print("✓ número total de registros preservado")
print("✓ ambas as classes presentes em treino e teste")


### 22. Configuração da validação cruzada estratificada

A atividade exige **validação cruzada de 5 dobras**.

Usaremos:

```python
StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
```

`StratifiedKFold` preserva aproximadamente a proporção das classes em cada dobra de treino e validação.


In [ ]:
N_SPLITS = 5

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print(cv)


### 23. Inspeção das 5 dobras

Antes de treinar os modelos, verificaremos o tamanho e a distribuição de spam em cada fold.

O conjunto de **teste final não participa dessas dobras**.


In [ ]:
fold_rows = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train), start=1):
    y_fold_train = y_train.iloc[train_idx]
    y_fold_valid = y_train.iloc[valid_idx]

    fold_rows.append({
        "fold": fold,
        "n_treino": len(train_idx),
        "n_validacao": len(valid_idx),
        "spam_treino_%": y_fold_train.mean() * 100,
        "spam_validacao_%": y_fold_valid.mean() * 100,
    })

fold_summary = pd.DataFrame(fold_rows)

display(
    fold_summary.style.format({
        "spam_treino_%": "{:.2f}%",
        "spam_validacao_%": "{:.2f}%",
    })
)


### 24. Interpretação metodológica

A partir deste ponto:

- `X_train` e `y_train` serão usados para **comparar e selecionar modelos**;
- `cv` definirá as **5 dobras estratificadas**;
- `X_test` e `y_test` permanecerão reservados para a **avaliação final**;
- nenhum resultado do conjunto de teste será usado para escolher modelo, hiperparâmetros ou limiar.

Essa separação ajuda a evitar uma estimativa excessivamente otimista do desempenho de generalização.

> Observação: a análise inicial identificou linhas duplicadas na base original. Elas serão mantidas no experimento principal para preservar os 4.601 registros do enunciado. Essa característica será registrada como limitação metodológica e poderá ser explorada posteriormente em análise de sensibilidade.


### Checkpoint T5-A3

Ao finalizar esta etapa no Google Colab, confirme:

- [ ] treino e teste foram criados;
- [ ] as proporções de spam e legítimo permanecem semelhantes nos dois conjuntos;
- [ ] não há sobreposição de índices;
- [ ] as 5 dobras apresentam proporções de classe semelhantes;
- [ ] o conjunto de teste final continua intocado.

**Próxima etapa — T5-A4:** comparar pelo menos três modelos vistos em aula usando as mesmas 5 dobras e reportar **média e desvio-padrão** das métricas.


---
## T5-A4 — Comparação de modelos com validação cruzada

O item 2 do trabalho solicita **pelo menos três modelos**, utilizando divisão estratificada e **validação cruzada de 5 dobras**, com reporte de **média e desvio-padrão**.

Nesta etapa compararemos quatro modelos clássicos:

1. **Regressão Logística**
2. **Árvore de Decisão**
3. **k-NN**
4. **Random Forest**

Para garantir uma comparação metodologicamente correta:

- todos os modelos usarão as **mesmas 5 dobras** definidas em `cv`;
- a avaliação será feita **somente em `X_train` e `y_train`**;
- `X_test` e `y_test` continuarão isolados para a avaliação final;
- algoritmos sensíveis à escala, como Regressão Logística e k-NN, usarão `StandardScaler` dentro de um `Pipeline`, evitando vazamento de informação entre as dobras.

> O experimento específico pedido no item 3 — **k-NN com e sem padronização** — será aprofundado na T5-A5. Aqui utilizaremos a versão padronizada do k-NN como representante dessa família.


### 25. Imports para modelagem e avaliação cruzada


In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier


### 26. Definição dos modelos

Os hiperparâmetros serão mantidos próximos aos valores padrão, pois o objetivo desta etapa é **comparar famílias de modelos**, e não realizar tuning extensivo.

A padronização fica **dentro do pipeline**, portanto é ajustada novamente em cada fold somente com os dados de treino daquela dobra.


In [ ]:
models = {
    "Regressão Logística": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE,
        )),
    ]),

    "Árvore de Decisão": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
    ),

    "k-NN padronizado": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(
            n_neighbors=5,
        )),
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

for name, model in models.items():
    print(f"{name}:")
    print(model)
    print("-" * 80)


### 27. Métricas de avaliação

Além da **acurácia**, que é explicitamente pedida no trabalho, registraremos:

- `precision`: entre os e-mails classificados como spam, quantos realmente eram spam;
- `recall`: entre os spams reais, quantos foram detectados;
- `F1`: média harmônica entre precision e recall;
- `ROC-AUC`: capacidade de separar as duas classes considerando diferentes limiares.

Essas métricas serão úteis porque, neste problema, falsos positivos e falsos negativos possuem consequências diferentes.


In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

print("Métricas:", ", ".join(scoring.keys()))


### 28. Validação cruzada dos quatro modelos

Cada modelo será avaliado nas **mesmas 5 dobras estratificadas**.

Para cada métrica serão preservados os resultados individuais dos folds e, em seguida, calcularemos:

\[
\text{média} = \frac{1}{5}\sum_{i=1}^{5} x_i
\]

e o respectivo **desvio-padrão** entre as cinco avaliações.


In [ ]:
cv_raw_results = {}
fold_records = []

for model_name, model in models.items():
    print(f"Avaliando: {model_name}...")

    scores = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    cv_raw_results[model_name] = scores

    for fold in range(N_SPLITS):
        fold_records.append({
            "modelo": model_name,
            "fold": fold + 1,
            "accuracy": scores["test_accuracy"][fold],
            "precision": scores["test_precision"][fold],
            "recall": scores["test_recall"][fold],
            "f1": scores["test_f1"][fold],
            "roc_auc": scores["test_roc_auc"][fold],
            "fit_time_s": scores["fit_time"][fold],
        })

print("✓ Validação cruzada concluída para todos os modelos.")


### 29. Resultados por fold

A tabela abaixo permite verificar não apenas a média final, mas também a **estabilidade** de cada modelo entre as cinco dobras.


In [ ]:
cv_folds_df = pd.DataFrame(fold_records)

display(
    cv_folds_df.style.format({
        "accuracy": "{:.4f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "roc_auc": "{:.4f}",
        "fit_time_s": "{:.3f}",
    })
)


### 30. Média e desvio-padrão das métricas

Esta é a tabela principal para atender ao item 2 do trabalho.

O desvio-padrão ajuda a avaliar a estabilidade: valores menores indicam que o desempenho variou menos entre os folds.


In [ ]:
metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]

summary_rows = []

for model_name in models.keys():
    model_folds = cv_folds_df[cv_folds_df["modelo"] == model_name]

    row = {"modelo": model_name}

    for metric in metrics:
        row[f"{metric}_media"] = model_folds[metric].mean()
        row[f"{metric}_dp"] = model_folds[metric].std(ddof=1)

    row["fit_time_media_s"] = model_folds["fit_time_s"].mean()
    summary_rows.append(row)

cv_summary = pd.DataFrame(summary_rows).sort_values(
    by="accuracy_media",
    ascending=False,
).reset_index(drop=True)

display(
    cv_summary.style.format({
        col: "{:.4f}"
        for col in cv_summary.columns
        if col != "modelo"
    })
)


### 31. Tabela compacta — média ± desvio-padrão

Para facilitar a leitura e posterior uso no relatório, apresentaremos as métricas no formato:

**média ± desvio-padrão**


In [ ]:
compact_rows = []

for _, row in cv_summary.iterrows():
    compact_rows.append({
        "modelo": row["modelo"],
        "accuracy": f'{row["accuracy_media"]:.4f} ± {row["accuracy_dp"]:.4f}',
        "precision": f'{row["precision_media"]:.4f} ± {row["precision_dp"]:.4f}',
        "recall": f'{row["recall_media"]:.4f} ± {row["recall_dp"]:.4f}',
        "f1": f'{row["f1_media"]:.4f} ± {row["f1_dp"]:.4f}',
        "roc_auc": f'{row["roc_auc_media"]:.4f} ± {row["roc_auc_dp"]:.4f}',
    })

cv_compact = pd.DataFrame(compact_rows)

display(cv_compact)


### 32. Comparação com o baseline

O baseline da T5-A2 obteve aproximadamente **60,60% de acurácia**, mas `recall = 0` para spam.

A célula abaixo compara diretamente a acurácia média dos modelos da validação cruzada com esse baseline.


In [ ]:
baseline_accuracy_reference = baseline_accuracy

comparison_to_baseline = cv_summary[
    ["modelo", "accuracy_media", "accuracy_dp"]
].copy()

comparison_to_baseline["ganho_abs_sobre_baseline"] = (
    comparison_to_baseline["accuracy_media"] - baseline_accuracy_reference
)

display(
    comparison_to_baseline.style.format({
        "accuracy_media": "{:.4f}",
        "accuracy_dp": "{:.4f}",
        "ganho_abs_sobre_baseline": "{:+.4f}",
    })
)


### 33. Interpretação dos resultados

Nesta etapa, **não escolheremos ainda o modelo final apenas pela maior acurácia**.

A análise deve considerar conjuntamente:

- desempenho médio;
- variação entre folds;
- precision e recall da classe spam;
- F1;
- ROC-AUC;
- custo computacional;
- e, principalmente, o requisito posterior do trabalho: **no máximo 1% dos e-mails legítimos podem ser marcados como spam**.

Esse último requisito torna a seleção do limiar operacional tão importante quanto a comparação entre os classificadores.

Também é importante lembrar que o conjunto `X_test` / `y_test` permanece sem uso nesta etapa.


### Checkpoint T5-A4

Ao executar esta etapa no Google Colab, registre:

- [ ] resultados dos 5 folds para cada modelo;
- [ ] média e desvio-padrão da acurácia;
- [ ] média e desvio-padrão de precision, recall, F1 e ROC-AUC;
- [ ] comparação com o baseline;
- [ ] confirmação de que o conjunto de teste final continua intocado.

**Próxima etapa — T5-A5:** comparar explicitamente o **k-NN com e sem padronização dos atributos**, usando as mesmas 5 dobras, e explicar tecnicamente a diferença.
